# MNIST: Trajectory Embedding (standalone Colab version)

Self-contained Colab copy of the exp4 pipeline — it deliberately re-implements
the few `core/` functions it needs so the single notebook can run on a GPU
runtime without the repository. The local equivalent is `run_growth.py`.

Pipeline: pretrained DDPM (`1aurent/ddpm-mnist`) → batched Euler integration
of the probability-flow ODE → trajectory embedding → $C(X) = \log\det(I+K)$.

**Part 1:** within-class vs between-class trajectory distance as a function of $t$.

**Part 2:** $C(X)$ vs $N$ for single-class (digit 0) and multi-class (cycling 0–9)
samples, with Beta($a$, $b$) time weighting.

Before running, upload `ddpm_mnist_pretrained.pt`
(produced by `python export_checkpoint.py`).

In [ ]:
!pip install diffusers accelerate -q

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from diffusers import UNet2DModel
from scipy.spatial.distance import pdist, squareform
from scipy.stats import beta as beta_dist

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load pretrained model
ckpt = torch.load('ddpm_mnist_pretrained.pt', map_location=device, weights_only=False)
unet = UNet2DModel(**{k: v for k, v in ckpt['unet_config'].items()
                      if k not in ('_class_name', '_diffusers_version', '_name_or_path',
                                   '_use_default_values')})
unet.load_state_dict(ckpt['unet_state_dict'])
unet = unet.to(device).eval()

# Noise schedule
cfg = ckpt['scheduler_config']
T = cfg['num_train_timesteps']
betas = np.linspace(cfg['beta_start'], cfg['beta_end'], T)
alphas = 1.0 - betas
alpha_bars = np.cumprod(alphas)

def t_to_k(t):
    return max(0, min(T - 1, int(round(t * (T - 1)))))

def sigma(t):
    return np.sqrt(1.0 - alpha_bars[t_to_k(t)])

def beta_continuous(t):
    return T * (cfg['beta_start'] + (cfg['beta_end'] - cfg['beta_start']) * t)

print(f'Model: {sum(p.numel() for p in unet.parameters()):,} params, T={T}')

In [ ]:
# Load MNIST, group by digit
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
ds = datasets.MNIST('data', train=True, download=True, transform=transform)

by_digit = {d: [] for d in range(10)}
for img, label in ds:
    by_digit[label].append(img.numpy().flatten())
by_digit = {d: np.array(v) for d, v in by_digit.items()}
print('Loaded MNIST.')

In [ ]:
@torch.no_grad()
def batched_ode_trajectories(x_flat, t_eval, n_euler_steps=200):
    """
    Batched Euler integration of VP probability flow ODE.
    Returns trajectories: (N, L, 784)
    """
    N, D = x_flat.shape
    L = len(t_eval)
    dt = 1.0 / n_euler_steps
    t_grid = np.linspace(0, 1.0, n_euler_steps + 1)
    record_at = [int(np.argmin(np.abs(t_grid - te))) for te in t_eval]

    z = torch.from_numpy(x_flat.reshape(N, 1, 28, 28).astype(np.float32)).to(device)
    trajectories = np.zeros((N, L, D), dtype=np.float32)

    for step in range(n_euler_steps):
        if step in record_at:
            j = record_at.index(step)
            trajectories[:, j, :] = z.reshape(N, D).cpu().numpy()

        t = t_grid[step]
        k = t_to_k(t)
        ts = torch.full((N,), k, device=device, dtype=torch.long)
        eps = unet(z, ts).sample
        s = -eps / sigma(t)
        z = z + (-0.5 * beta_continuous(t) * (z + s)) * dt

    if n_euler_steps in record_at:
        j = record_at.index(n_euler_steps)
        trajectories[:, j, :] = z.reshape(N, D).cpu().numpy()

    return trajectories

In [ ]:
def trajectory_embedding(trajectories, weights):
    N, L, D = trajectories.shape
    w = weights / weights.sum()
    weighted = trajectories * np.sqrt(w)[np.newaxis, :, np.newaxis]
    return weighted.reshape(N, L * D)

def calibrate_bandwidth(emb):
    D = squareform(pdist(emb))
    upper = D[np.triu_indices_from(D, k=1)]
    med_sq = np.median(upper ** 2)
    return 1.0 / med_sq if med_sq > 0 else 1.0

def compute_complexity(emb, lambda_):
    D = squareform(pdist(emb))
    K = np.exp(-lambda_ * D ** 2)
    N = K.shape[0]
    L = np.linalg.cholesky(np.eye(N) + K)
    return 2.0 * np.sum(np.log(np.diag(L)))

## Experiment 1: Trajectory Distance vs Time

Mean ||z₁(t) − z₂(t)|| for within-class and between-class pairs at each time t.

In [ ]:
# 5 images per digit = 50 total
N_per = 5
imgs_all = np.concatenate([by_digit[d][:N_per] for d in range(10)])
labs_all = np.array([d for d in range(10) for _ in range(N_per)])

L_fine = 30
t_fine = np.linspace(0.02, 0.98, L_fine)

print(f'Computing trajectories for {len(imgs_all)} images at {L_fine} time points...')
trajs_all = batched_ode_trajectories(imgs_all, t_fine, n_euler_steps=300)
print('Done.')

In [ ]:
within_mean = []
between_mean = []

for j in range(L_fine):
    z = trajs_all[:, j, :]
    w, b = [], []
    for i1 in range(len(labs_all)):
        for i2 in range(i1 + 1, len(labs_all)):
            d = np.linalg.norm(z[i1] - z[i2])
            (w if labs_all[i1] == labs_all[i2] else b).append(d)
    within_mean.append(np.mean(w))
    between_mean.append(np.mean(b))

within_mean = np.array(within_mean)
between_mean = np.array(between_mean)
ratio_curve = between_mean / within_mean

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(t_fine, within_mean, '-', color='steelblue', lw=2, label='Within-class')
axes[0].plot(t_fine, between_mean, '-', color='firebrick', lw=2, label='Between-class')
axes[0].set_xlabel('t', fontsize=12)
axes[0].set_ylabel('Mean $\\|z_1(t) - z_2(t)\\|$', fontsize=12)
axes[0].set_title('Trajectory distance vs noise level')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_fine, ratio_curve, '-', color='darkgreen', lw=2)
axes[1].set_xlabel('t', fontsize=12)
axes[1].set_ylabel('Between / Within', fontsize=12)
axes[1].set_title('Separation ratio')
axes[1].axhline(1, color='gray', ls='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('exp1_distance_vs_time.png', dpi=150)
plt.show()

print(f'Max ratio: {ratio_curve.max():.4f} at t={t_fine[np.argmax(ratio_curve)]:.3f}')

## Experiment 2: Complexity vs N

C(X) growth for single-class (digit 0) vs multi-class (cycling 0–9).

Time weighting via Beta(a, b) PDF — adjust `BETA_A`, `BETA_B` below.

In [ ]:
# === PARAMETERS ===
BETA_A = 2.0    # Beta pdf shape a
BETA_B = 2.0    # Beta pdf shape b
L = 10           # number of time evaluation points
N_EULER = 200    # Euler integration steps
N_VALUES = [5, 10, 15, 20, 30, 40, 50]

t_eval = np.linspace(0.05, 0.95, L)
weights = beta_dist.pdf(t_eval, BETA_A, BETA_B)
weights = weights / weights.sum()

# Show weighting
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.bar(t_eval, weights, width=0.04, color='steelblue', alpha=0.7)
ax.set_xlabel('t')
ax.set_ylabel('w(t)')
ax.set_title(f'Time weighting: Beta({BETA_A}, {BETA_B})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calibrate lambda from held-out zeros
print('Calibrating lambda...')
ref = by_digit[0][200:230]
ref_trajs = batched_ode_trajectories(ref, t_eval, n_euler_steps=N_EULER)
ref_emb = trajectory_embedding(ref_trajs, weights)
lambda_ = calibrate_bandwidth(ref_emb)
print(f'lambda = {lambda_:.6f}')

# Growth experiment
C_single = []
C_multi = []

for N in N_VALUES:
    # Single class (zeros)
    x_s = by_digit[0][:N]
    emb_s = trajectory_embedding(
        batched_ode_trajectories(x_s, t_eval, N_EULER), weights)
    c_s = compute_complexity(emb_s, lambda_)
    C_single.append(c_s)

    # Multi-class (cycling 0,1,...,9)
    counts = {d: 0 for d in range(10)}
    x_m = []
    for i in range(N):
        d = i % 10
        x_m.append(by_digit[d][counts[d]])
        counts[d] += 1
    x_m = np.array(x_m)
    emb_m = trajectory_embedding(
        batched_ode_trajectories(x_m, t_eval, N_EULER), weights)
    c_m = compute_complexity(emb_m, lambda_)
    C_multi.append(c_m)

    print(f'N={N:3d}  C_single={c_s:.4f}  C_multi={c_m:.4f}  ratio={c_m/c_s:.3f}')

print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_VALUES, C_single, 'o-', color='steelblue', lw=2, label='Single class (digit 0)')
ax.plot(N_VALUES, C_multi, 's-', color='firebrick', lw=2, label='Multi-class (cycling 0\u20139)')
ax.set_xlabel('Sample size N', fontsize=12)
ax.set_ylabel('C(X)', fontsize=12)
ax.set_title(f'Complexity Growth — Beta({BETA_A}, {BETA_B}) weighting', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('exp2_complexity_vs_N.png', dpi=150)
plt.show()

print(f'{"N":>5} {"C_single":>10} {"C_multi":>10} {"ratio":>8}')
for i, N in enumerate(N_VALUES):
    r = C_multi[i] / C_single[i] if C_single[i] > 0 else float('inf')
    print(f'{N:>5} {C_single[i]:>10.4f} {C_multi[i]:>10.4f} {r:>8.3f}')